In [22]:
#!pip install mlflow
import pandas as pd
import numpy as np

#NETTOYAGE
import re
import string
import nltk
"""nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')"""
from nltk.corpus import stopwords
from gensim.utils import simple_preprocess
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

#FEATURES
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
#BERT
from transformers import DistilBertTokenizer, TFDistilBertForSequenceClassification
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
import mlflow.tensorflow

#TRACKING
import mlflow
import mlflow.sklearn

# Importation du jeu

In [2]:
columns_name = ['TARGET', 'id', 'date', '??', 'user', 'tweet']

df = pd.read_csv("Data/training.1600000.processed.noemoticon.csv", encoding='ISO-8859-1', names=columns_name)

print(df["tweet"][df["tweet"].isnull()])

print(df.head())
print(df)

Series([], Name: tweet, dtype: object)
   TARGET          id                          date        ??  \
0       0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1       0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2       0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
3       0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
4       0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   

              user                                              tweet  
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...  
1    scotthamilton  is upset that he can't update his Facebook by ...  
2         mattycus  @Kenichan I dived many times for the ball. Man...  
3          ElleCTF    my whole body feels itchy and like its on fire   
4           Karoli  @nationwideclass no, it's not behaving at all....  
         TARGET          id                          date        ??  \
0             0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY

# Nettoyage du texte

## Fonction

In [3]:
#Fonction pour nettoyer le texte
def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return ""
    
    #Passer en minuscule tout le texte
    texte = texte.lower()

    #Supprimer des éléments frequents dans des tweets, mais jugés ininteressants pour l'analyse
    texte = re.sub(r'<.*?>', '', texte)
    texte = re.sub(r'@\w+', '', texte)
    texte = re.sub(r'\d+', '', texte)
    texte = re.sub(r'https?://\S+|www\.\S+', '', texte)

    #Supprimer les ponctuactions
    texte = texte.translate(str.maketrans("","",string.punctuation))
    
    texte = re.sub(r'\s+', ' ', texte)

    #Initialisation
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    
    #TOKENISATION
    tokens = texte.split()

    #Parcours des tokens:
    clean_tokens = [
        lemmatizer.lemmatize(token) 
        for token in tokens
        if token not in stop_words and len(token) > 2
    ]
            
    #Renvoie des elements à joindre
    return " ".join(clean_tokens)



## Application

In [4]:
df_trt = df.copy()[['TARGET', 'id', 'date','user', 'tweet']]

df_trt['tweet_net'] = df_trt['tweet'].apply(nettoyer_texte)
print(df_trt['tweet'], df_trt['tweet_net'])

0          @switchfoot http://twitpic.com/2y1zl - Awww, t...
1          is upset that he can't update his Facebook by ...
2          @Kenichan I dived many times for the ball. Man...
3            my whole body feels itchy and like its on fire 
4          @nationwideclass no, it's not behaving at all....
                                 ...                        
1599995    Just woke up. Having no school is the best fee...
1599996    TheWDB.com - Very cool to hear old Walt interv...
1599997    Are you ready for your MoJo Makeover? Ask me f...
1599998    Happy 38th Birthday to my boo of alll time!!! ...
1599999    happy #charitytuesday @theNSPCC @SparksCharity...
Name: tweet, Length: 1600000, dtype: object 0          awww thats bummer shoulda got david carr third...
1          upset cant update facebook texting might cry r...
2               dived many time ball managed save rest bound
3                            whole body feel itchy like fire
4                                      be

# Featuring

## pre entrainement

In [5]:
#Definition des variables
X = df_trt['tweet_net']
y = df_trt['TARGET']
y = y.replace(4, 1)

#Separation : 80% train / 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## ML FLOW

### Modèles simples

In [6]:
mlflow.set_experiment("Test_modeles")

modeles = [
    {
        "nom": "Logistic_Regression",
        "modele": LogisticRegression(max_iter=1000, C=1.0),
        "params": {"max_iter": 1000, "C": 1.0}
    },
    {
        "nom": "Random_Forest",
        "modele": RandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1),
        "params": {"n_estimators": 50, "max_depth": 10}
    }
]

for config in modeles:
    with mlflow.start_run(run_name=config['nom']):
        
        print(f"--- Entraînement de : {config['nom']} ---")
        
        # 1. Création de la Pipeline (Transformation + Modèle)
        # TfidfVectorizer : max_features=5000 pour limiter la mémoire
        pipe = Pipeline([
            ('tfidf', TfidfVectorizer(max_features=5000)), 
            ('clf', config['modele'])
        ])
        
        # 2. Entraînement
        pipe.fit(X_train, y_train)
        
        # 3. Prédiction
        y_pred = pipe.predict(X_test)
        
        # 4. Calcul des métriques
        acc = accuracy_score(y_test, y_pred)
        print(f"Accuracy : {acc:.4f}")
        
        # 5. LOGGING MLFLOW
        # On enregistre les paramètres donnés au début
        mlflow.log_params(config['params'])
        
        # On enregistre la performance
        mlflow.log_metric("accuracy", acc)
        
        # On enregistre le modèle COMPLET (Pipeline)
        # C'est ce fichier qu'on chargera dans l'API Azure plus tard
        mlflow.sklearn.log_model(pipe, "model")

2026/01/16 12:42:17 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/01/16 12:42:17 INFO mlflow.store.db.utils: Updating database tables
2026/01/16 12:42:17 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/16 12:42:17 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/01/16 12:42:17 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/16 12:42:17 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/01/16 12:42:17 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or 

--- Entraînement de : Logistic_Regression ---


2026/01/16 12:42:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Accuracy : 0.7723
--- Entraînement de : Random_Forest ---


2026/01/16 12:43:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Accuracy : 0.7047


### Modèles plus avancées

#### BERT

In [19]:
#TOKENISER de DistilBERT
bert_token = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')


def bert_encode(texts, max_len=100):
    return bert_token(
        texts.tolist(),
        truncation=True, 
        padding=True, 
        max_length=max_len, 
        return_tensors='tf'
    )
#ECHANTILLONNAGE : réduction du jeu
print("ECHANTILLONNAGE : réduction du jeu")
train_encode = bert_encode(X_train[:2000])
test_encode = bert_encode(X_test[:2000])

#Transformation en Dataset
print("Transformation en Dataset")
train_dataset = tf.data.Dataset.from_tensor_slices((dict(train_encode), y_train[:2000])).shuffle(1000).batch(16)
test_dataset = tf.data.Dataset.from_tensor_slices((dict(test_encode), y_test[:2000])).batch(16)


ECHANTILLONNAGE : réduction du jeu
Transformation en Dataset


In [20]:
mlflow.set_experiment("Test_modeles")
mlflow.tensorflow.autolog()

print("RUN")
with mlflow.start_run(run_name="DistilBERT_FineTuning"):
    model_bert = TFDistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=1, use_safetensors=False)
    
    # On utilise un learning rate TRES FAIBLE (5e-5) pour ne pas "casser" le cerveau pré-entraîné
    optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)
    
    # Compilation (La loss est souvent gérée en interne par HuggingFace, mais on précise pour Keras)
    print("Compilation")
    model_bert.compile(optimizer=optimizer, loss=model_bert.compute_loss, metrics=['accuracy'])
    
    # 2 epochs suffisent généralement pour BERT
    print("Execution")
    model_bert.fit(train_dataset, validation_data=test_dataset, epochs=2)
    
    print("Fin")

RUN


Some layers from the model checkpoint at distilbert-base-uncased were not used when initializing TFDistilBertForSequenceClassification: ['vocab_transform', 'vocab_projector', 'activation_13', 'vocab_layer_norm']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFDistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['pre_classifier', 'classifier', 'dropout_79']
You should probably TRAIN this model on a down-stream task to be able to use i

Compilation
Execution


AttributeError: '_TensorBoard' object has no attribute '_implements_train_batch_hooks'

#### LSTM

In [24]:
# --- 1. PRÉPARATION DES DONNÉES (Specifique LSTM) ---
MAX_WORDS = 10000       # On garde les 10k mots les plus fréquents
MAX_LEN = 100           # On coupe les tweets à 100 mots

# A. Initialisation du Tokenizer
print("Token")
tokenizer = Tokenizer(num_words=MAX_WORDS)
# IMPORTANT : On n'apprend le vocabulaire QUE sur le train set pour éviter la fuite de données
tokenizer.fit_on_texts(X_train[:2000])

# B. Conversion Texte -> Séquence de nombres
X_train_seq = tokenizer.texts_to_sequences(X_train[:2000])
X_test_seq = tokenizer.texts_to_sequences(X_test[:2000])

# C. Padding (Remplissage pour avoir la même taille partout)
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN)
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN)

#Entrainement
mlflow.set_experiment("Test_modeles")
mlflow.tensorflow.autolog()

print("RUN")
with mlflow.start_run(run_name="LSTM_Keras"):
    
    # Construction du modèle
    model = Sequential()
    # Couche Embedding : transforme les entiers en vecteurs denses
    model.add(Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN))
    # Couche LSTM
    model.add(LSTM(64, dropout=0.2)) 
    # Couche de sortie
    model.add(Dense(1, activation='sigmoid'))

    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    
    print("Entraînement LSTM en cours...")
    model.fit(X_train_pad, y_train[:2000], 
              validation_data=(X_test_pad, y_test[:2000]),
              epochs=3, 
              batch_size=64)
    
    print("LSTM terminé et tracké sur MLflow.")

Token
RUN
Entraînement LSTM en cours...
Epoch 1/3


F:\Applications\Anaconda\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 75ms/step - accuracy: 0.5275 - loss: 0.6917 - val_accuracy: 0.5620 - val_loss: 0.6860
Epoch 2/3
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.6900 - loss: 0.6506 - val_accuracy: 0.6565 - val_loss: 0.6299
Epoch 3/3
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 70ms/step - accuracy: 0.8045 - loss: 0.4582 - val_accuracy: 0.6650 - val_loss: 0.6262
LSTM terminé et tracké sur MLflow.
